# Decode Adversarial Queries

Decodes the SQL queries and join orders for the 5 worst adversarial plans by absolute and relative difference.

In [19]:
import ast
import os
import sys
sys.path.insert(0, '..')

import lark
import networkx as nx
import pandas as pd
import duckdb
from networkx.algorithms import approximation

from optimization.codec.codec import HashProbeStackMachineCodec, JoinTreeBranch, JoinTreeLeaf
from oracle.adversarial_queries import get_predicate_graph
from workload.workloads import IMDB_WORKLOAD_SET

In [20]:
# Connect to DuckDB (needed for predicate value lookups when reconstructing SQL)
if not os.path.exists('../workload/job-complex/job-complex.duckdb'):
    print('WARNING: imdb.duckdb not found - SQL reconstruction will be skipped')
    con = None
else:
    con = duckdb.connect('../workload/job-complex/job-complex.duckdb')
    print('Connected to imdb.duckdb')

# Build the join predicate graph and codec using the same infrastructure
# used to generate the adversarial queries
predicate_graph = get_predicate_graph(IMDB_WORKLOAD_SET)
all_tables = list(sorted(predicate_graph.nodes))
codec = HashProbeStackMachineCodec(all_tables)
print(f'Tables in codec ({len(all_tables)}): {all_tables}')

Connected to imdb.duckdb
Tables in codec (21): ['aka_name', 'aka_title', 'cast_info', 'char_name', 'comp_cast_type', 'company_name', 'company_type', 'complete_cast', 'info_type', 'keyword', 'kind_type', 'link_type', 'movie_companies', 'movie_info', 'movie_info_idx', 'movie_keyword', 'movie_link', 'name', 'person_info', 'role_type', 'title']


In [21]:
# Load the adversarial query CSVs, sort descending (highest advantage = worst DuckDB default plan)
df_abs = pd.read_csv('imdb_adversarial_absolute.csv')
df_rel = pd.read_csv('imdb_adversarial_relative.csv')

top5_abs = df_abs.sort_values('absolute_advantage_ms', ascending=False).head(5).reset_index(drop=True)
top5_rel = df_rel.sort_values('relative_advantage', ascending=False).head(5).reset_index(drop=True)

print(f'Loaded {len(df_abs)} absolute queries and {len(df_rel)} relative queries')
print(f'\nTop 5 by absolute advantage:')
print(top5_abs[['absolute_advantage_ms', 'default_time_ms', 'generated_time_ms']].to_string())
print(f'\nTop 5 by relative advantage:')
print(top5_rel[['relative_advantage', 'default_time_ms', 'generated_time_ms']].to_string())

Loaded 999 absolute queries and 911 relative queries

Top 5 by absolute advantage:
   absolute_advantage_ms  default_time_ms  generated_time_ms
0          299604.973793         300000.0         395.026207
1          299596.058846         300000.0         403.941154
2          299594.428658         300000.0         405.571342
3          299593.933225         300000.0         406.066775
4          299592.132449         300000.0         407.867551

Top 5 by relative advantage:
   relative_advantage  default_time_ms  generated_time_ms
0          764.957027         300000.0         392.178893
1          745.468506         300000.0         402.431488
2          741.599965         300000.0         404.530764
3          740.999034         300000.0         404.858828
4          735.743847         300000.0         407.750607


In [22]:
# Grammar for parsing the query DSL (same as in query_diversity.ipynb)
parsing_grammar = """
start: selector+
selector: "(" ident " " filter* ")"
?ident: /[a-z0-9_]+/
filter: int_filter | str_filter
int_filter: "(" ident " " intop " " intval ")"
str_filter: "(" ident " " strop ")"

!intop: "<" | ">" | "=" | "!="
!strop: "most_popular" | "random"

!intval: "first" | "median" | "last" | "mode" | /[0-9]{4}/
"""
g = lark.Lark(parsing_grammar)


def get_int_value(table_name, field, value):
    """Look up a concrete integer predicate value from DuckDB."""
    if value == 'first':
        sql = f'SELECT {field} FROM {table_name} WHERE {field} IS NOT NULL ORDER BY {field} LIMIT 1'
    elif value == 'median':
        sql = f'SELECT median({field}) FROM {table_name}'
    elif value == 'last':
        sql = f'SELECT {field} FROM {table_name} WHERE {field} IS NOT NULL ORDER BY {field} DESC LIMIT 1'
    elif value == 'mode':
        sql = f'SELECT {field} FROM {table_name} WHERE {field} IS NOT NULL GROUP BY {field} ORDER BY count(*) DESC LIMIT 1'
    else:
        p = int(value) / 1000
        sql = f'SELECT quantile_disc({field}, {p}) FROM {table_name}'
    return str(con.sql(sql).df().to_numpy()[0][0])


def get_str_value(table_name, field, str_val):
    """Look up a concrete string predicate value from DuckDB."""
    if str_val == 'most_popular':
        sql = f'SELECT {field} FROM {table_name} GROUP BY {field} ORDER BY count(*) DESC LIMIT 1'
    elif str_val == 'random':
        sql = f'SELECT {field} FROM {table_name} USING SAMPLE 1'
    else:
        raise ValueError(f'Unknown str_val: {str_val}')
    return str(con.sql(sql).df().to_numpy()[0][0])


def get_dsl_tables(query_dsl):
    """Return the tables directly referenced in the DSL string."""
    t = g.parse(query_dsl)
    return [str(selector.children[0]) for selector in t.children]


def get_connected_tables(query_dsl):
    """Return all tables needed (direct + Steiner tree connectors)."""
    tables_involved = get_dsl_tables(query_dsl)
    subgraph = approximation.steiner_tree(predicate_graph, tables_involved)
    return sorted(set(subgraph.nodes) | set(tables_involved))


def _build_where_parts(query_dsl):
    """Parse DSL and return (where_clauses, connected_tables). Requires DuckDB."""
    t = g.parse(query_dsl)
    where_clause = []
    tables_involved = []

    for selector in t.children:
        table_name = str(selector.children[0])
        tables_involved.append(table_name)
        for filt in selector.children[1:]:
            filt = filt.children[0]
            if filt.data == 'int_filter':
                field = str(filt.children[0])
                operator = str(filt.children[1].children[0])
                value = str(filt.children[2].children[0])
                rvalue = get_int_value(table_name, field, value)
                where_clause.append(f'{table_name}.{field} {operator} {rvalue}')
            elif filt.data == 'str_filter':
                field = str(filt.children[0])
                str_val = str(filt.children[1].children[0])
                rvalue = get_str_value(table_name, field, str_val)
                where_clause.append(f"{table_name}.{field} = '{rvalue}'")

    subgraph = approximation.steiner_tree(predicate_graph, tables_involved)
    for e in subgraph.edges:
        where_clause.append(subgraph.edges[e]['label'])

    connected_tables = sorted(set(subgraph.nodes) | set(tables_involved))
    return where_clause, connected_tables


def dsl_to_sql(query_dsl):
    """Convert a query DSL string to full SQL with default (comma-separated) FROM clause."""
    where_clause, connected_tables = _build_where_parts(query_dsl)
    from_clause = ', '.join(connected_tables)
    where_str = '\n  AND '.join(where_clause)
    return f'SELECT count(*)\nFROM {from_clause}\nWHERE {where_str}'


def join_tree_sexp(tree):
    """Render a join tree as an S-expression: ((A JOIN B) JOIN C)."""
    if isinstance(tree, JoinTreeLeaf):
        return tree.table
    return f'({join_tree_sexp(tree.left)} JOIN {join_tree_sexp(tree.right)})'


def build_hinted_sql(query_dsl, plan_vector_str):
    """Decode plan and return (sexp, runnable SQL with optimizer disabled).

    The SQL uses SET disabled_optimizers = 'join_order,build_side_probe_side'
    (from oracle/adversarial_queries.py) to force DuckDB to respect the join order.
    """
    plan_vector = ast.literal_eval(plan_vector_str)
    tables = get_connected_tables(query_dsl)
    tree = codec.decode([(t, 1) for t in tables], plan_vector)

    sexp = join_tree_sexp(tree)
    join_clause = tree.to_join_clause()  # e.g. (A CROSS JOIN B) CROSS JOIN C

    where_clause, _ = _build_where_parts(query_dsl)
    where_str = '\n  AND '.join(where_clause)

    hinted_sql = (
        "SET disabled_optimizers = 'join_order,build_side_probe_side';\n"
        f"SELECT count(*)\nFROM {join_clause}\nWHERE {where_str}"
    )
    return sexp, hinted_sql


print('Helper functions defined.')


Helper functions defined.


In [23]:
print('=' * 80)
print('TOP 5 QUERIES BY ABSOLUTE ADVANTAGE')
print('(Largest absolute difference: DuckDB default plan vs. adversarial plan)')
print('=' * 80)

for i, row in top5_abs.iterrows():
    print(f'\n{"-" * 80}')
    print(f'Rank {i+1}')
    print(f'  Default time:    {row["default_time_ms"]:>12,.1f} ms')
    print(f'  Generated time:  {row["generated_time_ms"]:>12,.1f} ms')
    print(f'  Absolute adv:    {row["absolute_advantage_ms"]:>12,.1f} ms')
    print(f'  Relative adv:    {row["relative_advantage"]:>12,.1f}x')
    print(f'\nDSL query:\n  {row["query"]}')

    if con is not None:
        try:
            sexp, hinted_sql = build_hinted_sql(row['query'], row['generated_plan'])
            print(f'\nAdversarial join order (S-expression):\n  {sexp}')
            print(f'\nSQL with adversarial join order:\n  ' + hinted_sql.replace('\n', '\n  '))
        except Exception as e:
            print(f'\nAdversarial join order / hinted SQL: <error: {e}>')
    else:
        try:
            plan_vector = ast.literal_eval(row['generated_plan'])
            tables = get_connected_tables(row['query'])
            tree = codec.decode([(t, 1) for t in tables], plan_vector)
            print(f'\nAdversarial join order (S-expression):\n  {join_tree_sexp(tree)}')
        except Exception as e:
            print(f'\nAdversarial join order: <error: {e}>')


TOP 5 QUERIES BY ABSOLUTE ADVANTAGE
(Largest absolute difference: DuckDB default plan vs. adversarial plan)

--------------------------------------------------------------------------------
Rank 1
  Default time:       300,000.0 ms
  Generated time:         395.0 ms
  Absolute adv:       299,605.0 ms
  Relative adv:           759.4x

DSL query:
  (company_type )(complete_cast )(info_type (info random))(keyword )(movie_link )(name )(title )

Adversarial join order (S-expression):
  ((((((((((complete_cast JOIN movie_link) JOIN title) JOIN movie_companies) JOIN movie_keyword) JOIN name) JOIN person_info) JOIN company_type) JOIN keyword) JOIN movie_info) JOIN info_type)

SQL with adversarial join order:
  SET disabled_optimizers = 'join_order,build_side_probe_side';
  SELECT count(*)
  FROM (((((((((complete_cast CROSS JOIN movie_link) CROSS JOIN title) CROSS JOIN movie_companies) CROSS JOIN movie_keyword) CROSS JOIN name) CROSS JOIN person_info) CROSS JOIN company_type) CROSS JOIN keywor

In [24]:
print('=' * 80)
print('TOP 5 QUERIES BY RELATIVE ADVANTAGE')
print('(Largest ratio: DuckDB default plan time / adversarial plan time)')
print('=' * 80)

for i, row in top5_rel.iterrows():
    print(f'\n{"-" * 80}')
    print(f'Rank {i+1}')
    print(f'  Default time:    {row["default_time_ms"]:>12,.1f} ms')
    print(f'  Generated time:  {row["generated_time_ms"]:>12,.1f} ms')
    print(f'  Absolute adv:    {row["absolute_advantage_ms"]:>12,.1f} ms')
    print(f'  Relative adv:    {row["relative_advantage"]:>12,.1f}x')
    print(f'\nDSL query:\n  {row["query"]}')

    if con is not None:
        try:
            sexp, hinted_sql = build_hinted_sql(row['query'], row['generated_plan'])
            print(f'\nAdversarial join order (S-expression):\n  {sexp}')
            print(f'\nSQL with adversarial join order:\n  ' + hinted_sql.replace('\n', '\n  '))
        except Exception as e:
            print(f'\nAdversarial join order / hinted SQL: <error: {e}>')
    else:
        try:
            plan_vector = ast.literal_eval(row['generated_plan'])
            tables = get_connected_tables(row['query'])
            tree = codec.decode([(t, 1) for t in tables], plan_vector)
            print(f'\nAdversarial join order (S-expression):\n  {join_tree_sexp(tree)}')
        except Exception as e:
            print(f'\nAdversarial join order: <error: {e}>')


TOP 5 QUERIES BY RELATIVE ADVANTAGE
(Largest ratio: DuckDB default plan time / adversarial plan time)

--------------------------------------------------------------------------------
Rank 1
  Default time:       300,000.0 ms
  Generated time:         392.2 ms
  Absolute adv:       299,607.8 ms
  Relative adv:           765.0x

DSL query:
  (company_type )(complete_cast )(info_type (info random))(keyword )(movie_link )(name )(title )

Adversarial join order (S-expression):
  ((((((((((complete_cast JOIN movie_link) JOIN title) JOIN movie_companies) JOIN movie_keyword) JOIN name) JOIN person_info) JOIN company_type) JOIN keyword) JOIN movie_info) JOIN info_type)

SQL with adversarial join order:
  SET disabled_optimizers = 'join_order,build_side_probe_side';
  SELECT count(*)
  FROM (((((((((complete_cast CROSS JOIN movie_link) CROSS JOIN title) CROSS JOIN movie_companies) CROSS JOIN movie_keyword) CROSS JOIN name) CROSS JOIN person_info) CROSS JOIN company_type) CROSS JOIN keyword) CRO